In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd

OVERTURE_RELEASE = "2026-09-23.1"   # which Overture dataset build to load

from helpers.features import get_processed_dfs
from helpers.data_display import dis_pct_suc_on_mask
from process_dumb_models import get_model_results

In [ ]:
ot_train = pd.read_parquet(f'../Overture/Data/ct_pairs_train_{OVERTURE_RELEASE}.parquet')
ot_train, ot_train_label = ot_train.drop(columns = ["label"]), ot_train["label"]
ot_val = pd.read_parquet(f'../Overture/Data/ct_pairs_val_{OVERTURE_RELEASE}.parquet')
ot_val, ot_val_label = ot_val.drop(columns = ["label"]), ot_val["label"]
ot_test = pd.read_parquet(f'../Overture/Data/ct_pairs_test_{OVERTURE_RELEASE}.parquet')
ot_test, ot_test_label = ot_test.drop(columns = ["label"]), ot_test["label"]

# Split each address into house number + street remainder.
# Number token = starts with a digit, runs to the first space (12, 12A, 16-18, 20-A).
# extract and replace use the same token so the number never stays on the street.
# No number -> NaN in Address_* -> number_missing = True.
for df in [ot_train, ot_val, ot_test]:
    df["Address_a"] = df["address_a"].str.extract(r"^(\d[A-Z0-9a-z-]*)\s")
    df["StreetName_a"] = df["address_a"].str.replace(r"^(\d[A-Z0-9a-z-]*\s+)", '', regex = True)
    df["Address_b"] = df["address_b"].str.extract(r"^(\d[A-Z0-9a-z-]*)\s")
    df["StreetName_b"] = df["address_b"].str.replace(r"^(\d[A-Z0-9a-z-]*\s+)", '', regex = True)
    df["both_number_missing"] = (df["Address_a"].isna() & df["Address_b"].isna())
    df["number_missing"] = (df["Address_a"].isna() | df["Address_b"].isna()) & ~df["both_number_missing"]
    
# Same ZIP column names as Stamford, so one feature function serves both
ot_train = ot_train.rename(columns = {"zip_a": "ZipCode_a", "zip_b": "ZipCode_b"})
ot_val = ot_val.rename(columns = {"zip_a": "ZipCode_a", "zip_b": "ZipCode_b"})
ot_test = ot_test.rename(columns = {"zip_a": "ZipCode_a", "zip_b": "ZipCode_b"})

In [ ]:
# Full feature build (cached; ~1.5 min on first run)
train_overture, val_overture, test_overture = get_processed_dfs([
    [ot_train, "./overture_training/overture_evaluation_train.parquet"],
    [ot_val, "./overture_training/overture_evaluation_val.parquet"],
    [ot_test, "./overture_training/overture_evaluation_test.parquet"],
])

In [ ]:
# Overture -> Overture, scored on val
ret = get_model_results(train_overture, ot_train_label, val_overture, ot_val, ot_val_label)

In [ ]:
for model, (preds, probs) in ret.items():
    same_mask = (ot_val["Address_a"] == ot_val["Address_b"])
    dis_pct_suc_on_mask(ot_val_label, preds, same_mask, f"{model}")
    


### Results — Overture → Overture, val (2026-10-11)

23 features (22 similarities + `number_missing`), default hyperparameters, no class weighting.

| model | P (match) | R (match) | F1 (match) | case1 rejected | case2 rejected | case3 rejected |
|---|---|---|---|---|---|---|
| LogReg | 0.98 | 0.96 | 0.97 | 99.8% | 3% | 99.97% |
| RF | 0.97 | 0.97 | 0.97 | 99.5% | 18% | 99.9% |
| XGBoost | 0.97 | 0.97 | 0.97 | 99.5% | 21% | 99.9% |

~225 errors out of 41,865 val pairs per model. Read with care:
- **Headline likely inflated** by identical-string positives (~70% raw) → next: recall on identical vs non-identical positives.
- **Case 2** (directional variants, e.g. `N MAIN ST` / `S MAIN ST`) mostly accepted as matches — Step 1b's false-positive problem on real data; only 34 val rows.
- **Visible error sources:** text before the number on side B → `number_missing`; `ST`/`STREET` suffix variants.

Next: identical vs non-identical breakdown + errors by `number_missing`; Stamford → Overture; ZIP ablation.

*Diagnostic queries + results for this section: **Diagnostics** at the bottom of this notebook → Overture baseline.*

## Diagnostics

One-off checks kept out of the pipeline above. Paths are relative to this notebook's folder.

### House-number regex

**Problem:** The old `^(\d+)\s+` pattern missed numbers like `12A` and `16-18`, leaving them in the street name.  
**Check:** New regex on hand-picked cases; streets still starting with a digit; raw addresses the old pattern missed.  
**Result:** Hand cases correct; 473 / 492 streets start with a digit (ordinals, fractions, double numbers); old pattern missed 1,202 / 1,484 letter suffixes and 444 / 742 ranges (A / B) (re-run 2026-10-13)

```python
# Test the house-number token regex on hand-picked cases
check = pd.DataFrame({"address": ["12 MAIN ST", "12A MAIN ST", "16-18 ELM ST", "20-A ELM ST"]})
check["check"] = check["address"].str.extract(r"^(\d[A-Z0-9a-z-]*)\s")
print(check)

# After parsing: sample + count of streets that still start with a digit (ordinal streets, 1/2 addresses, double numbers)
ret = pd.concat([ot_train, ot_val, ot_test])
print(ret[ret["StreetName_a"].str.match(r"^\d")][["StreetName_a"]].sample(frac=1).head(10))
print(ret["StreetName_a"].str.match(r"^\d").sum())
print(ret[ret["StreetName_b"].str.match(r"^\d")][["StreetName_b"]].sample(frac=1).head(10))
print(ret["StreetName_b"].str.match(r"^\d").sum())

# Raw addresses the old ^(\d+)\s+ pattern missed: letter suffix (12A) on A / B, then ranges (16-18) on A / B
print(ret["address_a"].str.match(r"^\d+[A-Za-z]+\s").sum())
print(ret["address_b"].str.match(r"^\d+[A-Za-z]+\s").sum())
print(ret["address_a"].str.match(r"^[0-9]+-[0-9]+\s").sum())
print(ret["address_b"].str.match(r"^[0-9]+-[0-9]+\s").sum())
```

### Streets under 2 characters

**Problem:** The Overlap bigram metric divides by zero on strings shorter than 2 characters.  
**Check:** Find any short street names.  
**Result:** Test split only: street `1` (7 on A, 11 on B) → guarded in `features.py` (re-run 2026-10-13)

```python
# Streets under 2 characters (break bigram metrics): only the test addresses that are literally "1"
for df in [ot_train, ot_val, ot_test]:
    print(df[df["StreetName_a"].str.len() < 2]["StreetName_a"])
    print(df[df["StreetName_b"].str.len() < 2]["StreetName_b"])
    print()
```

### Positives with identical addresses

**Problem:** Suspected the models were misclassifying positives whose addresses are identical.  
**Check:** Misclassifications within `case0`, split by identical vs. different address and by number status.  
**Result:** Identical: LR 1 / 3,390 wrong (a ZIP disagreement), RF 0, XGB 0. Different: 82–85% correct. The `number_missing` subset is empty by construction since `both_number_missing` was split out; use `both_number_missing` instead (re-run 2026-10-13)

```python
print([k for k in ret.keys()])
mask = ((ot_val["neg_type"] == "case0") & (ot_val["address_a"] == ot_val["address_b"])).values
totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
print([int(c) for c in totals])

# missing = [ot_val[mask][(preds != ot_val_label) & (ot_val["number_missing"] == False)] for model, (preds, probs) in ret.items()]
# for miss,model in zip(missing,[k for k in ret.keys()]):
#     print(f"{'='*20} {model} {'='*20}")
#     print(miss[["address_a", "address_b", "ZipCode_a", "ZipCode_b"]])


# mask = ((ot_val["neg_type"] == "case0") & (ot_val["address_a"] != ot_val["address_b"])).values
# totals = [(preds[mask] == ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([int(c) for c in totals])

# mask = ((ot_val["neg_type"] == "case0") & 
#         (ot_val["address_a"] == ot_val["address_b"]) & 
#         (ot_val['number_missing'] == True)).values
# n = mask.sum()
# totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([float(c/n) for c in totals])

# mask = ((ot_val["neg_type"] == "case0") & 
#         (ot_val["address_a"] == ot_val["address_b"]) & 
#         (ot_val['number_missing'] == False)).values
# n = mask.sum()
# totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([float(c/n) for c in totals])

# print(
# len(ot_val[
#     (ot_val["neg_type"] == "case0") & 
#     (ot_val["address_a"] == ot_val["address_b"]) & 
#     (ot_val['number_missing'] == True)
# ]))
```
